# BDA Mini‑Project (Colab + PySpark, NoSQL, Streaming, R export)

In [ ]:
# === Cell 1: Minimal dependencies (Colab) ===
!pip -q install pyspark==3.5.1 pandas numpy pymongo mmh3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 2.6 MB/s eta 0:00:00


In [ ]:
# === Cell 2: Spark session init ===
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("bda-mini").getOrCreate()
spark

## Generate a tiny dataset (synthetic)
Created two CSVs:
- `clicks.csv`: timestamp, user_id, session_id, event (view/cart/buy), product_id, category, price
- `orders.csv`: order_id, user_id, ts, product_id, qty, price

In [ ]:
# === Cell 3: Data generator (50k clicks, 5k orders) ===
import pandas as pd, numpy as np, os, time
rng = np.random.default_rng(7)
n_users = 2000
n_products = 800
n_clicks = 50000
n_orders = 5000
cats = np.array(["electronics","books","clothing","home","sports"])
prices = np.round(rng.uniform(5, 300, n_products),2)

prod_ids = np.arange(1, n_products+1)
prod_cat = rng.choice(cats, size=n_products)

def synth_clicks(n):
    ts_start = pd.Timestamp('2025-08-01')
    ts = ts_start + pd.to_timedelta(rng.integers(0, 60*60*24*30, n), unit='s')
    user = rng.integers(1, n_users+1, n)
    session = user*1000 + rng.integers(1, 100, n)
    pid = rng.choice(prod_ids, n)
    event = rng.choice(["view","view","view","cart","buy"], n)  # skewed
    cat = prod_cat[pid-1]
    price = prices[pid-1]
    return pd.DataFrame({"ts":ts, "user_id":user, "session_id":session, "event":event,
                         "product_id":pid, "category":cat, "price":price})

def synth_orders(n):
    ts_start = pd.Timestamp('2025-08-01')
    ts = ts_start + pd.to_timedelta(rng.integers(0, 60*60*24*30, n), unit='s')
    user = rng.integers(1, n_users+1, n)
    pid = rng.choice(prod_ids, n)
    qty = rng.integers(1, 3, n)
    price = prices[pid-1]
    oid = np.arange(1, n+1)
    return pd.DataFrame({"order_id":oid, "user_id":user, "ts":ts, "product_id":pid, "qty":qty, "price":price})

os.makedirs('/content/data', exist_ok=True)
clicks = synth_clicks(n_clicks)
orders = synth_orders(n_orders)
clicks.to_csv('/content/data/clicks.csv', index=False)
orders.to_csv('/content/data/orders.csv', index=False)
clicks.head(3), orders.head(3)

(                   ts  user_id  session_id event  product_id     category  \
 0 2025-08-23 00:51:58     1966     1966012  view         137  electronics   
 1 2025-08-03 12:23:13      534      534056  view         341       sports   
 2 2025-08-05 10:40:01      678      678050   buy         387  electronics   
 
     price  
 0  294.22  
 1   97.57  
 2  264.85  ,
    order_id  user_id                  ts  product_id  qty   price
 0         1      796 2025-08-24 03:21:47         192    2  172.28
 1         2     1182 2025-08-17 10:43:22         756    2   88.94
 2         3      935 2025-08-29 01:17:53         584    2   97.98)

In [ ]:
# === Cell 4: Load CSVs into Spark DataFrames ===
clicks_s = spark.read.option("header","true").option("inferSchema","true").csv("/content/data/clicks.csv")
orders_s = spark.read.option("header","true").option("inferSchema","true").csv("/content/data/orders.csv")
print(clicks_s.count(), orders_s.count())
clicks_s.printSchema()
orders_s.printSchema()

50000 5000
root
 |-- ts: timestamp (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- event: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- qty: integer (nullable = true)
 |-- price: double (nullable = true)



## MapReduce with Spark transformations
Relational-algebra operations and a simple matrix‑vector style co‑occurrence.

In [ ]:
# === Cell 5: Selection, Projection, Union/Intersection/Difference ===
from pyspark.sql.functions import col

# Selection: only 'buy' events
buys = clicks_s.filter(col("event") == "buy").select("user_id","product_id","price")
print("#buys:", buys.count())

# Projection: choose columns
proj = clicks_s.select("user_id","product_id","category")

# Union: clicks users ∪ orders users
click_users = clicks_s.select(col("user_id").alias("uid")).distinct()
order_users = orders_s.select(col("user_id").alias("uid")).distinct()
u_union = click_users.union(order_users).distinct()
u_inter = click_users.intersect(order_users)
u_diff = click_users.subtract(order_users)
print("Union, Intersect, Diff:", u_union.count(), u_inter.count(), u_diff.count())

#buys: 9925
Union, Intersect, Diff: 2000 1855 145


In [ ]:
# === Cell 6: Item co-occurrence (within session) ===
from pyspark.sql.functions import collect_set, explode, asc, desc
from itertools import combinations

# Get the set of products viewed per session
views = clicks_s.filter(col("event")=="view").groupBy("session_id").agg(collect_set("product_id").alias("items"))

# Create pair counts (i,j) from each session's item set
def pair_rows(items):
    items = sorted(items)
    for i,j in combinations(items, 2):
        yield (i,j,1)

pairs = views.rdd.flatMap(lambda row: pair_rows(row.items)).toDF(["i","j","c"])
pair_counts = pairs.groupBy("i","j").sum("c").withColumnRenamed("sum(c)","count")
pair_counts.orderBy(desc("count")).show(10)

# For simplicity produce top-5 similar items per item using raw co-view counts
from pyspark.sql.window import Window
import pyspark.sql.functions as F
w = Window.partitionBy("i").orderBy(F.desc("count"))
topk = pair_counts.withColumn("rk", F.row_number().over(w)).filter(col("rk")<=5)
topk.show(10)

+---+---+-----+
|  i|  j|count|
+---+---+-----+
| 15|764|    2|
|155|720|    2|
| 52|504|    2|
|230|614|    2|
|701|762|    1|
|105|332|    1|
|162|399|    1|
| 45|363|    1|
| 90|590|    1|
| 80|580|    1|
+---+---+-----+
only showing top 10 rows

+---+---+-----+---+
|  i|  j|count| rk|
+---+---+-----+---+
|  1|588|    1|  1|
|  1|471|    1|  2|
|  1|393|    1|  3|
|  1| 64|    1|  4|
|  1|583|    1|  5|
|  2|157|    1|  1|
|  2|483|    1|  2|
|  2|625|    1|  3|
|  2|181|    1|  4|
|  2|466|    1|  5|
+---+---+-----+---+
only showing top 10 rows



## Save aggregates for R / NoSQL

In [ ]:
# === Cell 7: Product aggregates & exports ===
import pyspark.sql.functions as F
prod_agg = clicks_s.groupBy("product_id","category").agg(
    F.sum(F.when(col("event")=="view",1).otherwise(0)).alias("views"),
    F.sum(F.when(col("event")=="cart",1).otherwise(0)).alias("carts"),
    F.sum(F.when(col("event")=="buy",1).otherwise(0)).alias("buys"),
).withColumn("conv_rate", (F.col("buys")/(F.col("views")+F.lit(1))))

prod_agg.orderBy(F.desc("views")).show(5)
outdir = "/content/out"
prod_agg.coalesce(1).write.mode("overwrite").option("header","true").csv(f"{outdir}/product_agg")
topk.coalesce(1).write.mode("overwrite").option("header","true").csv(f"{outdir}/item_topk")

+----------+--------+-----+-----+----+-------------------+
|product_id|category|views|carts|buys|          conv_rate|
+----------+--------+-----+-----+----+-------------------+
|       435|   books|   60|   11|  11|0.18032786885245902|
|       164|    home|   57|   10|   9|0.15517241379310345|
|       440|   books|   57|   14|  12|0.20689655172413793|
|       222|    home|   53|   20|  13|0.24074074074074073|
|       128|   books|   53|   12|  12| 0.2222222222222222|
+----------+--------+-----+-----+----+-------------------+
only showing top 5 rows



In [ ]:
# === Cell 8: Local persistence fallback ===
import os, json
outdir = "/content/out"
os.makedirs(outdir, exist_ok=True)

# Write product aggregates and item-topk to JSON & CSV
prod_agg_pdf = prod_agg.limit(500).toPandas()
item_topk_pdf = topk.toPandas()

prod_json = f"{outdir}/product_agg.json"
topk_json  = f"{outdir}/item_topk.json"
prod_csv   = f"{outdir}/product_agg.csv"
topk_csv   = f"{outdir}/item_topk.csv"

prod_agg_pdf.to_json(prod_json, orient="records")
item_topk_pdf.to_json(topk_json, orient="records")
prod_agg_pdf.to_csv(prod_csv, index=False)
item_topk_pdf.to_csv(topk_csv, index=False)

print("Wrote:")
print(" -", prod_json)
print(" -", topk_json)
print(" -", prod_csv)
print(" -", topk_csv)

# Quick “NoSQL-like” read demo: get top-5 products by views
import pandas as pd
check = pd.read_json(prod_json)
print("Top-5 by views:")
print(check.sort_values("views", ascending=False).head(5)[["product_id","category","views","buys","conv_rate"]])


Wrote:
 - /content/out/product_agg.json
 - /content/out/item_topk.json
 - /content/out/product_agg.csv
 - /content/out/item_topk.csv
Top-5 by views:
     product_id     category  views  buys  conv_rate
253         440        books     57    12   0.206897
435         164         home     57     9   0.155172
483         222         home     53    13   0.240741
315         610        books     53     7   0.129630
287         182  electronics     51    11   0.211538


## Streaming with Bloom, Flajolet–Martin (FM), and DGIM
Simulating a live stream by appending lines to a folder; Spark Structured Streaming reads from it.
Each line format: `ts,user_id,event,product_id`.

In [ ]:
# === Cell 9: Helper data structures ===
import mmh3, math, time, pathlib
from pyspark.sql.functions import split, col

class Bloom:
    def __init__(self, m=1024, k=4):
        self.m = m; self.k = k; self.bits = [0]*m
    def _idx(self, x, i):
        return abs(mmh3.hash(str(x), seed=i)) % self.m
    def add(self, x):
        for i in range(self.k): self.bits[self._idx(x,i)] = 1
    def contains(self, x):
        return all(self.bits[self._idx(x,i)] for i in range(self.k))

class FM:
    # Flajolet–Martin using multiple hash functions, estimate distinct
    def __init__(self, k=16):
        self.k = k
        self.R = [0]*k
    @staticmethod
    def _rho(x):
        # position of rightmost 1-bit in hash
        h = abs(mmh3.hash(str(x)))
        if h==0: return 32
        r=0
        while (h & 1)==0:
            r+=1; h >>= 1
        return r
    def add(self, x):
        for i in range(self.k):
            r = self._rho(mmh3.hash(str(x), seed=i))
            if r > self.R[i]: self.R[i] = r
    def estimate(self):
        phi = 0.77351
        return int((2 ** (sum(self.R)/self.k)) / phi)

class DGIM:
    # Sliding-window count of 1s over last W items
    def __init__(self, W=256):
        self.W = W
        self.buckets = []  # list of (size, timestamp)
        self.t = 0
    def add(self, bit):
        self.t += 1
        # drop old
        cutoff = self.t - self.W
        self.buckets = [(s,ts) for (s,ts) in self.buckets if ts>cutoff]
        if bit==1:
            self.buckets.append((1,self.t))
            # merge
            i=0
            while True:
                same = [idx for idx,(s,_) in enumerate(self.buckets) if s==(1<<i)]
                if len(same) <= 2: break
                # merge oldest two
                same.sort(key=lambda idx:self.buckets[idx][1])
                i1,i2 = same[0],same[1]
                ts = max(self.buckets[i1][1], self.buckets[i2][1])
                # remove larger index first
                for j in sorted([i1,i2], reverse=True): self.buckets.pop(j)
                self.buckets.append(((1<<(i+1)), ts))
                i+=1
    def estimate(self):
        if not self.buckets: return 0
        # sum sizes, subtract half of oldest bucket
        self.buckets.sort(key=lambda x:x[1], reverse=True)
        total = 0
        for i,(s,ts) in enumerate(self.buckets):
            if i==len(self.buckets)-1: total += s//2
            else: total += s
        return total

bloom = Bloom(m=2048,k=5)
fm = FM(k=16)
dgim = DGIM(W=256)
print("Stream structures ready.")

Stream structures ready.


In [ ]:
# === Cell 10: Prepare stream directory and a small writer ===
import os, pandas as pd
stream_dir = "/content/stream_dir"
os.makedirs(stream_dir, exist_ok=True)

# seed initial mini-batch
seed = clicks.sample(200)[["ts","user_id","event","product_id"]].copy()
seed.to_csv(os.path.join(stream_dir, f"batch_0.csv"), index=False, header=False)
print("Seed stream written to", stream_dir)

Seed stream written to /content/stream_dir


In [ ]:
# === Cell 11: Start Spark Structured Streaming and update FM/Bloom/DGIM ===
from pyspark.sql.functions import input_file_name

spark.conf.set("spark.sql.streaming.schemaInference", True)
raw = spark.readStream.format("csv").option("header","false").load(stream_dir)
parsed = raw.select(
    col("_c0").alias("ts"),
    col("_c1").cast("int").alias("user_id"),
    col("_c2").alias("event"),
    col("_c3").cast("int").alias("product_id")
)

def process_batch(df, epoch):
    rows = df.select("user_id","event").collect()
    report = {"n":len(rows), "added_bloom":0, "fm_est":None, "dgim_est":None}
    for r in rows:
        uid = r[0]
        if not bloom.contains(uid):
            bloom.add(uid); report["added_bloom"] += 1
        fm.add(uid)
        dgim.add(1 if r[1]=="buy" else 0)
    report["fm_est"] = fm.estimate()
    report["dgim_est"] = dgim.estimate()
    print(f"[epoch {epoch}] batch={report['n']} newBF={report['added_bloom']} FM≈{report['fm_est']} DGIM≈{report['dgim_est']}")

q = parsed.writeStream.outputMode("append").format("noop").foreachBatch(process_batch).start()
print("Streaming started. To append more data, run the next cell multiple times.")

Streaming started. To append more data, run the next cell multiple times.


In [ ]:
# === Cell 12: Append more events to the stream (run multiple times) ===
batch_id = len([f for f in os.listdir(stream_dir) if f.startswith('batch_')])
more = clicks.sample(200)[["ts","user_id","event","product_id"]]
more.to_csv(os.path.join(stream_dir, f"batch_{batch_id}.csv"), index=False, header=False)
print(f"Appended batch_{batch_id}.csv with {len(more)} rows.")

Appended batch_1.csv with 200 rows.


In [ ]:
# === Cell 13: Stop streaming when done ===
q.stop()
print("Streaming stopped.")

Streaming stopped.


##Simple Recommendations
Top‑N similar items using co‑views and get user‑wise recs from their last views/buys.

In [ ]:
# === Cell 14: Recommend top‑N ===
import pyspark.sql.functions as F

last_actions = clicks_s.filter(col("event").isin(["view","buy"]))\
    .groupBy("user_id").agg(F.max("ts").alias("last_ts"))\
    .join(clicks_s, on=["user_id"])\
    .filter(col("ts")==col("last_ts"))\
    .select("user_id","product_id").distinct()

recs = last_actions.alias("la").join(topk.alias("tk"), col("la.product_id")==col("tk.i"), "left")\
    .groupBy("la.user_id").agg(F.collect_list("tk.j").alias("candidates"))

recs.show(5, truncate=False)

+-------+--------------------+
|user_id|candidates          |
+-------+--------------------+
|1      |[]                  |
|2      |[480, 426, 187, 661]|
|3      |[]                  |
|4      |[278, 464]          |
|5      |[753, 708]          |
+-------+--------------------+
only showing top 5 rows



##Export for R (DAU, category stats)

Creates small CSVs

In [ ]:
# === Cell 15: Export CSVs for R ===
from pyspark.sql.functions import to_date
dau = clicks_s.select(to_date(col("ts")).alias("d"))\
        .groupBy("d").count().withColumnRenamed("count","dau")
cat_stats = prod_agg.groupBy("category").agg(F.sum("views").alias("views"), F.sum("buys").alias("buys"))

dau.coalesce(1).write.mode("overwrite").option("header",True).csv(f"{outdir}/r_dau")
cat_stats.coalesce(1).write.mode("overwrite").option("header",True).csv(f"{outdir}/r_cat_stats")
print("Exported to /content/out/r_dau and /content/out/r_cat_stats")

Exported to /content/out/r_dau and /content/out/r_cat_stats
